# VGG16 and ResNet50 Image Classification

**Transfer learning for cats, dogs, and horses**

This notebook trains and evaluates VGG16 and ResNet50 on the prepared dataset, compares validation performance, and displays confusion matrices and classification reports.

Dataset folders required: `dataset/train/{cats,dogs,horses}` and `dataset/validation/{cats,dogs,horses}`. Images are resized to 224 × 224 pixels.

## 1. Import libraries

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support
from IPython.display import display

print('TensorFlow:', tf.__version__)
print('GPUs detected:', len(tf.config.list_physical_devices('GPU')))

## 2. Set paths and load dataset

In [ ]:
BASE_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
TRAIN_DIR = os.path.join(BASE_DIR, 'dataset', 'train')
VAL_DIR = os.path.join(BASE_DIR, 'dataset', 'validation')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
PLOT_DIR = os.path.join(OUTPUT_DIR, 'plots')
CM_DIR = os.path.join(OUTPUT_DIR, 'confusion_matrices')
RESULT_DIR = os.path.join(OUTPUT_DIR, 'results')
for folder in [PLOT_DIR, CM_DIR, RESULT_DIR]:
    os.makedirs(folder, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 5
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=True, seed=SEED)
val_ds = tf.keras.utils.image_dataset_from_directory(VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False)
class_names = train_ds.class_names
num_classes = len(class_names)
if num_classes < 3:
    raise ValueError(f'Expected at least 3 classes, found {class_names}')
print('Classes:', class_names)
print('Training images:', sum(len(files) for _, _, files in os.walk(TRAIN_DIR)))
print('Validation images:', sum(len(files) for _, _, files in os.walk(VAL_DIR)))
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## 3. View sample training images

In [ ]:
images, labels = next(iter(train_ds))
plt.figure(figsize=(10, 7))
for i in range(min(9, len(images))):
    ax = plt.subplot(3, 3, i + 1)
    ax.imshow(tf.cast(images[i], tf.uint8).numpy())
    ax.set_title(class_names[int(np.argmax(labels[i].numpy()))])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Build transfer-learning models

In [ ]:
def build_model(model_name, num_classes):
    if model_name == 'VGG16':
        base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        preprocess = vgg_preprocess
    elif model_name == 'ResNet50':
        base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        preprocess = resnet_preprocess
    else:
        raise ValueError('Unknown model name')
    base.trainable = False
    inputs = layers.Input(shape=(224, 224, 3))
    x = layers.Lambda(preprocess, name=f'{model_name.lower()}_preprocessing')(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs, outputs, name=f'{model_name}_classifier')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ImageNet pretrained convolutional layers are frozen; the classification head is trained.

## 5. Train and evaluate both models

Run this cell to train both models for five epochs. Pretrained ImageNet weights may need to download on the first run.

In [ ]:
def evaluate_model(model, model_name):
    y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in val_ds])
    probabilities = model.predict(val_ds, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)
    report = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    accuracy = float(np.mean(y_true == y_pred))
    print(f'\n{model_name} validation accuracy: {accuracy:.4f}')
    print(report)
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(values_format='d')
    plt.title(f'{model_name} Confusion Matrix')
    plt.tight_layout()
    plt.savefig(os.path.join(CM_DIR, f'{model_name.lower()}_confusion_matrix.png'), dpi=150)
    plt.show()
    with open(os.path.join(RESULT_DIR, f'{model_name.lower()}_classification_report.txt'), 'w', encoding='utf-8') as f:
        f.write(report)
    return {'model': model_name, 'validation_accuracy': accuracy, 'precision_macro': float(precision), 'recall_macro': float(recall), 'f1_macro': float(f1), 'epochs': EPOCHS}

histories, results, trained_models = {}, [], {}
for model_name in ['VGG16', 'ResNet50']:
    print('\n' + '=' * 50 + '\nTraining ' + model_name)
    model = build_model(model_name, num_classes)
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
    histories[model_name] = history
    trained_models[model_name] = model
    model.save(os.path.join(OUTPUT_DIR, f'{model_name.lower()}_model.keras'))
    results.append(evaluate_model(model, model_name))
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation accuracy')
    plt.title(f'{model_name} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training loss')
    plt.plot(history.history['val_loss'], label='Validation loss')
    plt.title(f'{model_name} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f'{model_name.lower()}_training_history.png'), dpi=150)
    plt.show()

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(RESULT_DIR, 'model_comparison.csv'), index=False)
with open(os.path.join(RESULT_DIR, 'model_comparison.json'), 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print('Saved results to:', RESULT_DIR)
display(results_df)

## 6. Compare model metrics

In [ ]:
ax = results_df.set_index('model')[['validation_accuracy', 'precision_macro', 'recall_macro', 'f1_macro']].plot(kind='bar', figsize=(10, 5), ylim=(0, 1), rot=0)
ax.set_title('VGG16 vs ResNet50')
ax.set_ylabel('Score')
ax.set_xlabel('Model')
plt.tight_layout()
plt.show()
best_model_name = results_df.sort_values('validation_accuracy', ascending=False).iloc[0]['model']
print('Best model by validation accuracy:', best_model_name)

## 7. Predict a new image

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array

def predict_image(image_path, model_name=None):
    if model_name is None:
        model_name = best_model_name
    image = load_img(image_path, target_size=IMG_SIZE)
    arr = img_to_array(image)
    probabilities = trained_models[model_name].predict(np.expand_dims(arr, axis=0), verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))
    print('Model:', model_name)
    print('Predicted class:', class_names[predicted_index])
    print('Confidence:', f'{probabilities[predicted_index] * 100:.2f}%')
    return class_names[predicted_index], probabilities

# Example: replace the filename with an image in test_images.
# predict_image(os.path.join(BASE_DIR, 'test_images', 'your_image.jpg'))

## 8. Conclusion

Compare validation accuracy, macro precision, macro recall, and macro F1-score. The model with stronger validation metrics performed better on this validation split. The dataset is small, so performance may vary with a larger and more diverse dataset.

Outputs include the model comparison CSV, classification reports, training plots, confusion matrices, and saved Keras models in the outputs folder.